# Building an Always-on Commercial Decision System with AI Agents

This walkthrough builds the governed agent in the same order as the chapter: signal, request, technical foundation, model, tools, memory, loop, guardrails, evaluation, workbench, and the later outcome. It runs without an API key. All data is synthetic.

![The completed Roventra decision workbench showing the released recommendation, 10 numbered system areas, and operating controls for a saved later-decision case.](assets/figures/figure_16_1_workbench_overview.png)

*Figure 16.1: The completed decision workbench. Numbered areas identify the signal, run state, model mode, evidence, memory, loop, guardrails, evaluation gate, human authority, and outcome path.*

| Highlight | Section | Function |
| --- | --- | --- |
| 1 | 16.1 | Signal intake and confirmed request |
| 2 | 16.2 | Case, run, state, and current node |
| 3 | 16.3 | Saved, mock, or live model mode |
| 4 | 16.4 | Governed evidence and tool status |
| 5 | 16.5 | Checkpoint, history, and reopened run |
| 6 | 16.6 | Loop progress and revisions |
| 7 | 16.7 | Validation, limits, and pause reason |
| 8 | 16.8 | Evaluation release gate |
| 9 | 16.9 | Human disposition controls |
| 10 | 16.10 | Outcome recording and reopening |

*Table 16.1: Workbench areas and the sections that build them.*

## 1. Environment and imports

In [1]:
import sys, tempfile
from datetime import date
from pathlib import Path

START = Path.cwd().resolve()
REPO_ROOT = next(
    path for path in (START, *START.parents)
    if (path / "ch16_decision" / "scripts").is_dir()
)
SCRIPT_DIR = REPO_ROOT / "ch16_decision" / "scripts"
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(SCRIPT_DIR))

from build_database import build
from signal_monitor import (
    evaluate_hcp_digital_signal, read_trigger, confirm_signal, default_case_id)
from runtime import AgentRuntime, build_decision_request
from memory import CaseStore
from models import HumanDisposition, OutcomeEvent
from config import LATER_DECISION_DATE

build()  # deterministic synthetic commercial environment
WORK = Path(tempfile.mkdtemp())
print("environment built; working directory ready")


environment built; working directory ready


## 2. Detect the monitored signal

In [2]:
readout = read_trigger(date(2026, 7, 14))
for line in readout.as_lines():
    print(line)
signal = evaluate_hcp_digital_signal(date(2026, 7, 14))
print('signal:', signal.signal_id, signal.status)

community clicks 2026-W23->2026-W27: 61 -> 124 (+103%)
rise >= 30%? True
recent claims maturity: 1% (< 80%? True)
weekly NRx growth: -12% (<= 5%? True)
candidate opens: True
signal: SIG-452ac8d116 candidate


![Five observed weekly engagement values lead to a reversible first action. A 12-week bracket ends at the matched-market result, followed by the later scale decision.](assets/figures/figure_16_2_signal_and_lag.png)

*Figure 16.2: Engagement is available on July 14. The matched-market result becomes available 12 weeks later, when the same case reopens for a scale decision.*

## 3. Confirm the typed decision request

In [3]:
case_id = default_case_id()
request = confirm_signal(signal, build_decision_request(
    'first', case_id=case_id, signal_id=signal.signal_id,
    evidence_date=signal.evidence_date))
print('case:', request.case_id)
print('proposed move:', request.proposed_move)

case: CASE-ROVENTRA-HCP-2026
proposed move: Shift about a quarter of quarterly DTC budget into HCP digital


## 4. Technical foundation and shared state

In [4]:
from decision_graph import DecisionState, build_graph
from agents import LLM
graph = build_graph(LLM(mock=True))
nodes = [n for n in graph.get_graph().nodes if not n.startswith('__')]
print('typed state fields:', len(DecisionState.model_fields))
print('graph nodes:', len(nodes))

typed state fields: 27
graph nodes: 10


![A layered cutaway of the agent runtime connects the signal, specialized roles, model, tools, memory, limits, and human disposition through one typed-state spine.](assets/figures/figure_16_3_agent_anatomy.png)

*Figure 16.3: Typed state forms the runtime spine. The model context contains approved summaries and excludes restricted records.*

## 5. Connect the model (mock plumbing check)

In [5]:
import agents
llm = LLM(mock=True)
framing = agents.frame_decision(llm, request, 'first')
print('summary:', framing.decision_summary)
print('requested tools:', len(framing.requested_tools))
print('metered:', llm.drain_usage()[0].model_id)

summary: [MOCK] Frame whether the DTC-to-HCP-digital move is justified.
requested tools: 5
metered: claude-haiku-4-5 [MOCK]


## 6. Tools, reviewed SQL, and cited evidence

In [6]:
from tools import TOOL_CATALOG, run_tool
approved_tools = [name for name in framing.requested_tools
                  if name in TOOL_CATALOG]
evidence = [item for name in approved_tools
            for item in run_tool(name, 'first')]
print('approved and executed:', len(approved_tools))
print('typed evidence records:', len(evidence))
print('first citation:', evidence[0].citation)

approved and executed: 5
typed evidence records: 5
first citation: mmm_channel_results (model mmm_v4.2)


In [7]:
import json
from data_access import DATA_DIR, query_approved_data
reviewed = json.loads((DATA_DIR.parent / 'generated_outputs' /
    'ch16_reviewed_ad_hoc_query.json').read_text())
result = query_approved_data(reviewed['sql'])
print('review status:', reviewed['review_status'])
print('columns:', ', '.join(result.columns))
for row in result.rows:
    print(row)

review status: approved_after_sql_review
columns: segment, access_state, engaged_hcp_count, engagement_events, mature_nrx
('academic', 'stable', 8, 276, 4)
('academic', 'unstable', 8, 260, 5)
('community', 'stable', 24, 1027, 14)
('community', 'unstable', 8, 349, 4)


![A typed experiment request passes schema, allow-list, availability-date, and read-only checks before returning cited evidence and a separate audit record.](assets/figures/figure_16_4_governed_tool_call.png)

*Figure 16.4: A typed tool request passes argument, allow-list, date, and read-only checks. The return is one `EvidenceRecord` with the estimate, uncertainty, and stable citation.*

## 7. Durable checkpoint and restart recovery

In [8]:
runtime = AgentRuntime(mock=True, store=CaseStore(WORK / 'cases.sqlite'),
                       checkpoint_path=WORK / 'ckpt.sqlite')
runtime.create_case(signal, request)
status = runtime.start_run(case_id, mode='mock')
print('paused before:', status.next_node)
restarted = AgentRuntime(mock=True, store=CaseStore(WORK / 'cases.sqlite'),
                         checkpoint_path=WORK / 'ckpt.sqlite')
print('recovered:', restarted.get_run(status.run_id).analyst.selected_option_name)

paused before: human_approval
recovered: Reversible matched-market test


![An active run writes durable state into persistent memory. A checkpoint keyed by thread ID restores an interrupted run, while case history keyed by case ID reopens the decision weeks later.](assets/figures/figure_16_5_memory_across_two_dates.png)

*Figure 16.5: Persistent memory serves two time spans. A checkpoint keyed by `thread_id` restores an interrupted run; case history keyed by `case_id` reopens the later decision.*

## 8. The bounded loop and options

In [9]:
import math
from decision_services import simulate_budget_scenario
run = runtime.get_run(status.run_id)
for option, scenario in zip(run.option_set.options, run.scenarios):
    print(f'{option.name}: ${option.budget_moved_usd:,} feasible={scenario.feasible} '
          f'+{scenario.expected_incr_nrx_low}..{scenario.expected_incr_nrx_high} NRx')
test_option = next(option for option in run.option_set.options
                   if option.name == 'Reversible matched-market test')
budget, ceiling, scale, uncertainty = 187_500, 400, 210_000, 0.48
mid = round(ceiling * (1 - math.exp(-budget / scale)))
low = round(mid * (1 - uncertainty))
high = round(mid * (1 + uncertainty))
scenario = simulate_budget_scenario(test_option, 'first')
print('calculated range:', low, 'to', high,
      '| service range:', scenario.expected_incr_nrx_low,
      'to', scenario.expected_incr_nrx_high)

Reversible matched-market test: $187,500 feasible=True +123..349 NRx
Full requested move: $1,200,000 feasible=False +83..437 NRx
calculated range: 123 to 349 | service range: 123 to 349


![Ten declared LangGraph nodes carry shared decision state from frame through deliver. The main path runs through evidence gathering, integration, option design, simulation, selection, validation, and review.](assets/figures/figure_16_6_langgraph_nodes.png)

*Figure 16.6: Ten declared LangGraph nodes carry shared decision state from framing to delivery. Review can return the graph to `frame` or `propose_options`. The graph pauses before `human_approval`; `deliver` assembles the released decision record.*

## 9. Guardrails and the human interrupt

In [10]:
print('validation:', run.validation.status, '| reviewer:', run.review.disposition)
final = runtime.submit_disposition(status.run_id, HumanDisposition(
    decision='approve', reviewer='Brand lead', reason='Bounded, reversible, cited.'))
print('status:', final.status)

validation: pass | reviewer: pass
status: approved


## 10. Evaluation scorecard and live release gate

The committed live case results make the scorecard reproducible without another model call. The release function applies the same thresholds used by the harness.

In [11]:
from evaluation import CaseResult, score_suite
from evaluate_agent import passes_release_gate
for suite in ('development', 'holdout'):
    path = (DATA_DIR.parent / 'generated_outputs' /
            f'ch16_eval_live_{suite}_ch16_benchmark_v3_cases.json')
    rows = json.loads(path.read_text())
    metrics = score_suite([CaseResult(**row) for row in rows])
    gate = passes_release_gate(metrics, 'live', suite)
    print(f"{suite}: {metrics['cases_scored']} cases | "
          f"task {metrics['task_completion']:.1%} | "
          f"flagged {metrics['flagged_for_review_rate']:.1%} | "
          f"gate {'PASS' if gate else 'FAIL'}")

development: 23 cases | task 100.0% | flagged 4.3% | gate PASS


holdout: 4 cases | task 100.0% | flagged 50.0% | gate FAIL


## 11. Workbench service check

In [12]:
from fastapi.testclient import TestClient
from ch16_decision.app.app import create_app
web = Path(tempfile.mkdtemp())
with TestClient(create_app(web / 'c.sqlite', web / 'k.sqlite')) as client:
    print('health:', client.get('/health').json()['status'])
    print('workbench:', client.get('/?phase=first').status_code)

health: ok


workbench: 200


.venv/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 12. Later outcome and reopened case

In [13]:
runtime.ingest_outcome(case_id, OutcomeEvent(
    outcome_id='OUT-2026-1006-A1', case_id=case_id, decision_id='DEC-2026-0714-A1',
    available_date=LATER_DECISION_DATE, measurement_window='2026-W35..W40',
    observed_incremental_nrx=248, confidence_low=180, confidence_high=300,
    population='community', geography='US DMAs', source='prior_decisions',
    source_version='v1', maturity_status='mature'))
later = runtime.reopen_case(case_id, mode='mock')
runtime.submit_disposition(later.run_id, HumanDisposition(
    decision='approve', reviewer='Brand lead', reason='Scale the proven segment.'))
delivered = runtime.get_run(later.run_id)
print('later recommendation:', delivered.analyst.selected_option_name)
print('expected:', delivered.learning.expected_range)
print('observed:', delivered.learning.observed_result)

later recommendation: Staged community rollout
expected: 123 to 349 incremental NRx
observed: 248 incremental NRx


## Optional: one live model call

This cell runs only when `CH16_RUN_LIVE_NOTEBOOK=1` and `ANTHROPIC_API_KEY` are both set. It makes one real structured call and prints the model id and token usage.

In [14]:
import os
run_live = (os.environ.get('CH16_RUN_LIVE_NOTEBOOK') == '1'
            and bool(os.environ.get('ANTHROPIC_API_KEY')))
if run_live:
    live = LLM(mock=False)
    out = agents.frame_decision(live, request, 'first')
    usage = live.drain_usage()[0]
    print('live model:', usage.model_id, '| output tokens:', usage.output_tokens)
else:
    print('Live call skipped; set CH16_RUN_LIVE_NOTEBOOK=1 and an API key to run it.')

Live call skipped; set CH16_RUN_LIVE_NOTEBOOK=1 and an API key to run it.
